# 29 — Prioritize and validate LIANA communication axes

**Purpose:** reproduce the historical Notebook 26 post-processing on the canonical
global and condition-specific LIANA outputs from repository Notebooks 27 and 28.

**Historical counterpart:**
`26_LIANA_final_prioritized_axes_and_figures_HVG8000_r0_6.ipynb`

This notebook does **not** rerun LIANA. It reads fixed upstream TSV outputs,
validates their provenance, schemas, canonical labels, membership, and historical
checkpoints, then creates deterministic thesis-focused candidate tables and figures.

Run top to bottom in a fresh kernel.

# LIANA final prioritization and thesis-ready figures (HVG8000 r0.6)

This notebook does **not** rerun LIANA. It consolidates the canonical global and
condition-specific LIANA outputs into communication axes suitable for thesis reporting.

Main tasks:

1. Read the canonical global LIANA summaries from Notebook 27.
2. Read the canonical condition-specific LIANA summaries from Notebook 28.
3. Combine and prioritize biologically relevant communication axes.
4. Summarize candidate interactions including `PTN/MDK–PTPRZ1/NCL`,
   `TIMP1–CD63`, `LGALS1–ITGB1`, `MIF–EGFR`, `VIM–CD44`, `VEGFA`, and `APP–CD74`.
5. Create final thesis-ready tables and figures.

Interpretation note: this is not a replicate-aware statistical differential-communication
analysis. The outputs are prioritized candidate communication axes and should be interpreted
alongside cell annotation, CNV status, cell-state composition, and expression-program results.


In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import re
from datetime import datetime

pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 200)

import os

def _find_repository_root(start=None):
    """Locate the repository root or use GLIOMA_PROJECT_ROOT when set."""
    start = Path.cwd() if start is None else Path(start)
    start = start.expanduser().resolve()
    for candidate in (start, *start.parents):
        if (candidate / "README.md").exists() and (candidate / "code").exists():
            return candidate
    return start

PROJECT_ROOT = Path(
    os.environ.get("GLIOMA_PROJECT_ROOT", _find_repository_root())
).expanduser().resolve()
METADATA_DIR = PROJECT_ROOT / 'metadata'
FIGURE_DIR = PROJECT_ROOT / 'figures' / 'LIANA_final_prioritization_HVG8000_r0_6'
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

OUT_PREFIX = 'LIANA_final_prioritization_HVG8000_r0_6'

GLOBAL_PREFIX = 'LIANA_global_main_HVG8000_r0_6_clean_labels'
COND_PREFIX = 'LIANA_condition_specific_HVG8000_r0_6'

print('PROJECT_ROOT:', PROJECT_ROOT)
print('METADATA_DIR:', METADATA_DIR)
print('FIGURE_DIR:', FIGURE_DIR)

PROJECT_ROOT: G:\Repository\glioma-cnv-single-cell-atlas
METADATA_DIR: G:\Repository\glioma-cnv-single-cell-atlas\metadata
FIGURE_DIR: G:\Repository\glioma-cnv-single-cell-atlas\figures\LIANA_final_prioritization_HVG8000_r0_6


In [2]:
def read_tsv(path, required=False):
    path = Path(path)
    if not path.exists():
        msg = f'[MISSING] {path}'
        if required:
            raise FileNotFoundError(msg)
        print(msg)
        return pd.DataFrame()
    df = pd.read_csv(path, sep='\t')
    print(f'[READ] {path.name}: {df.shape}')
    return df


def write_tsv(df, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    if path.exists():
        path.unlink()
    df.to_csv(path, sep='\t', index=False)
    print(f'[SAVED] {path} shape={df.shape}')
    return path


def save_fig(path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    if path.exists():
        path.unlink()
    plt.tight_layout()
    plt.savefig(path, dpi=220, bbox_inches='tight')
    plt.close()
    print(f'[SAVED FIG] {path}')
    return path


def safe_col(df, candidates):
    for c in candidates:
        if c in df.columns:
            return c
    return None


def rank_col(df):
    for c in ['rank_for_sorting', 'primary_rank', 'magnitude_rank', 'specificity_rank', 'median_rank', 'median_primary_rank', 'min_rank', 'min_primary_rank']:
        if c in df.columns:
            return c
    return None


def clean_lr_pair(row):
    if 'lr_pair' in row and pd.notna(row['lr_pair']):
        return str(row['lr_pair'])
    lig = str(row.get('ligand', row.get('ligand_complex', '')))
    rec = str(row.get('receptor', row.get('receptor_complex', '')))
    return f'{lig}→{rec}'


def simplify_lr_pair(lr):
    lr = str(lr)
    lr = lr.replace('_', '-') if lr.count('→') == 0 else lr
    return lr


def axis_display_name(axis):
    if pd.isna(axis):
        return 'unknown'
    axis = str(axis)
    mapping = {
        'malignant_or_cycling_to_malignant_or_cycling': 'Malignant/cycling → malignant/cycling',
        'malignant_or_cycling_to_microenvironment': 'Malignant/cycling → microenvironment',
        'microenvironment_to_malignant_or_cycling': 'Microenvironment → malignant/cycling',
        'microenvironment_to_microenvironment': 'Microenvironment → microenvironment',
        'malignant_to_microenvironment': 'Malignant/cycling → microenvironment',
        'microenvironment_to_malignant': 'Microenvironment → malignant/cycling',
    }
    return mapping.get(axis, axis.replace('_', ' → '))

## 1. Read LIANA global and condition-specific summary files

In [3]:
# Read only the canonical upstream outputs used by this notebook.
global_cell_counts = read_tsv(METADATA_DIR / f'{GLOBAL_PREFIX}_celltype_counts_used.tsv', required=True)
global_axis = read_tsv(METADATA_DIR / f'{GLOBAL_PREFIX}_axis_summary.tsv', required=True)
global_bio_axis = read_tsv(METADATA_DIR / f'{GLOBAL_PREFIX}_biological_axis_summary.tsv', required=True)
global_lr_summary = read_tsv(METADATA_DIR / f'{GLOBAL_PREFIX}_ligand_receptor_summary.tsv', required=True)
global_run = read_tsv(METADATA_DIR / f'{GLOBAL_PREFIX}_run_summary.tsv', required=True)
global_consistency = read_tsv(METADATA_DIR / f'{GLOBAL_PREFIX}_notebook26_consistency_audit.tsv', required=True)

cond_run = read_tsv(METADATA_DIR / f'{COND_PREFIX}_run_summary.tsv', required=True)
cond_axis = read_tsv(METADATA_DIR / f'{COND_PREFIX}_axis_summary.tsv', required=True)
cond_bio_axis = read_tsv(METADATA_DIR / f'{COND_PREFIX}_biological_axis_summary.tsv', required=True)
cond_presence = read_tsv(METADATA_DIR / f'{COND_PREFIX}_prioritized_interaction_condition_presence.tsv', required=True)
cond_prioritized = read_tsv(METADATA_DIR / f'{COND_PREFIX}_prioritized_interactions.tsv', required=True)
cond_canonical_summary = read_tsv(METADATA_DIR / f'{COND_PREFIX}_canonical_run_summary.tsv', required=True)
cond_aggregate_audit = read_tsv(METADATA_DIR / f'{COND_PREFIX}_aggregate_result_checkpoint_audit.tsv', required=True)

EXPECTED_GLOBAL_TYPES = {
    'Malignant_AC_like_reactive_malignant',
    'Malignant_NPC_neuronal_like_malignant',
    'Malignant_OPC_NPC_like_malignant',
    'Malignant_cycling_candidate',
    'Malignant_hypoxia_stress_malignant',
    'Endothelial', 'Mural_Pericyte_Fibroblast',
    'Oligodendrocyte_myelinating', 'B_Plasma',
}
EXPECTED_CONDITION_TYPES = EXPECTED_GLOBAL_TYPES - {'B_Plasma'}
EXPECTED_CONDITIONS = {'GBM', 'LGG', 'ndGBM', 'rGBM'}

required_columns = {
    'global_cell_counts': (global_cell_counts, {'liana_celltype', 'observed_n_cells', 'match'}),
    'global_axis': (global_axis, {'communication_axis', 'n_prioritized_interactions'}),
    'global_bio_axis': (global_bio_axis, {'biological_axis', 'n_interactions'}),
    'global_lr_summary': (global_lr_summary, {'ligand_complex', 'receptor_complex', 'lr_pair', 'min_primary_rank'}),
    'cond_run': (cond_run, {'condition', 'status', 'n_cells', 'n_celltypes', 'n_liana_rows', 'n_prioritized'}),
    'cond_axis': (cond_axis, {'condition', 'axis', 'n_interactions'}),
    'cond_bio_axis': (cond_bio_axis, {'condition', 'biological_axis', 'n_interactions'}),
    'cond_presence': (cond_presence, {'interaction_key', 'source_clean', 'target_clean', 'lr_pair', 'n_conditions', 'conditions'}),
    'cond_prioritized': (cond_prioritized, {'condition', 'interaction_key', 'source_clean', 'target_clean', 'lr_pair', 'magnitude_rank'}),
}
for name, (frame, columns) in required_columns.items():
    missing = columns - set(frame.columns)
    if missing:
        raise AssertionError(f'{name}: missing required columns {sorted(missing)}')

global_types = set(global_cell_counts['liana_celltype'].astype(str))
condition_types = set(cond_prioritized['source_clean'].astype(str)) | set(cond_prioritized['target_clean'].astype(str))
all_output_labels = global_types | condition_types
forbidden_labels = sorted(
    label for label in all_output_labels
    if label.startswith('NonMalignant_')
    or 'Malignant_non_malignant' in label
    or label.startswith('Review_')
)

global_run_map = dict(zip(global_run['item'].astype(str), global_run['value']))
cond_summary_map = dict(zip(cond_canonical_summary['item'].astype(str), cond_canonical_summary['value']))

input_checks = [
    ('global_celltypes_exact', global_types == EXPECTED_GLOBAL_TYPES, len(global_types), len(EXPECTED_GLOBAL_TYPES)),
    ('condition_celltypes_exact', condition_types == EXPECTED_CONDITION_TYPES, len(condition_types), len(EXPECTED_CONDITION_TYPES)),
    ('canonical_labels_only', len(forbidden_labels) == 0, ';'.join(forbidden_labels), ''),
    ('global_cell_counts_match', global_cell_counts['match'].astype(str).str.lower().eq('true').all(), True, True),
    ('global_prioritized_rows', str(global_run_map.get('n_prioritized_rows')) == '1384', global_run_map.get('n_prioritized_rows'), 1384),
    ('global_unique_lr_pairs', len(global_lr_summary) == 264, len(global_lr_summary), 264),
    ('global_consistency_all_match', global_consistency['match'].astype(str).str.lower().eq('true').all(), True, True),
    ('condition_set_exact', set(cond_run['condition'].astype(str)) == EXPECTED_CONDITIONS, ';'.join(sorted(cond_run['condition'].astype(str))), ';'.join(sorted(EXPECTED_CONDITIONS))),
    ('condition_runs_completed', cond_run['status'].astype(str).eq('completed').all(), int(cond_run['status'].astype(str).eq('completed').sum()), 4),
    ('condition_prioritized_rows', len(cond_prioritized) == 4300, len(cond_prioritized), 4300),
    ('condition_presence_rows', len(cond_presence) == 2450, len(cond_presence), 2450),
    ('condition_prioritized_threshold', pd.to_numeric(cond_prioritized['magnitude_rank']).le(0.05).all(), True, True),
    ('condition_interaction_keys_unique_within_condition', not cond_prioritized.duplicated(['condition', 'interaction_key']).any(), int(cond_prioritized.duplicated(['condition', 'interaction_key']).sum()), 0),
    ('presence_interaction_keys_unique', not cond_presence['interaction_key'].duplicated().any(), int(cond_presence['interaction_key'].duplicated().sum()), 0),
    ('condition_aggregate_checkpoints', cond_aggregate_audit['match'].astype(str).str.lower().eq('true').all(), True, True),
    ('condition_canonical_summary_complete', str(cond_summary_map.get('n_conditions_completed')) == '4', cond_summary_map.get('n_conditions_completed'), 4),
]
input_audit = pd.DataFrame(input_checks, columns=['check', 'match', 'observed', 'expected'])
write_tsv(input_audit, METADATA_DIR / f'{OUT_PREFIX}_input_audit.tsv')
display(input_audit)
if not input_audit['match'].all():
    raise AssertionError('Notebook 29 canonical input audit failed.')

readiness = pd.DataFrame([
    {'item': 'postprocessing_only', 'value': True},
    {'item': 'liana_rerun', 'value': False},
    {'item': 'global_input_notebook', 'value': 27},
    {'item': 'condition_input_notebook', 'value': 28},
    {'item': 'global_cell_counts_rows', 'value': len(global_cell_counts)},
    {'item': 'global_prioritized_lr_pairs', 'value': len(global_lr_summary)},
    {'item': 'condition_specific_runs', 'value': len(cond_run)},
    {'item': 'condition_specific_prioritized_rows', 'value': len(cond_prioritized)},
    {'item': 'condition_presence_rows', 'value': len(cond_presence)},
    {'item': 'canonical_input_audit_all_match', 'value': bool(input_audit['match'].all())},
])
write_tsv(readiness, METADATA_DIR / f'{OUT_PREFIX}_readiness.tsv')
display(readiness)
display(cond_run)

[READ] LIANA_global_main_HVG8000_r0_6_clean_labels_celltype_counts_used.tsv: (9, 7)
[READ] LIANA_global_main_HVG8000_r0_6_clean_labels_axis_summary.tsv: (4, 5)
[READ] LIANA_global_main_HVG8000_r0_6_clean_labels_biological_axis_summary.tsv: (25, 7)
[READ] LIANA_global_main_HVG8000_r0_6_clean_labels_ligand_receptor_summary.tsv: (264, 8)
[READ] LIANA_global_main_HVG8000_r0_6_clean_labels_run_summary.tsv: (13, 2)
[READ] LIANA_global_main_HVG8000_r0_6_clean_labels_notebook26_consistency_audit.tsv: (5, 3)
[READ] LIANA_condition_specific_HVG8000_r0_6_run_summary.tsv: (4, 11)
[READ] LIANA_condition_specific_HVG8000_r0_6_axis_summary.tsv: (16, 7)
[READ] LIANA_condition_specific_HVG8000_r0_6_biological_axis_summary.tsv: (228, 7)
[READ] LIANA_condition_specific_HVG8000_r0_6_prioritized_interaction_condition_presence.tsv: (2450, 16)
[READ] LIANA_condition_specific_HVG8000_r0_6_prioritized_interactions.tsv: (4300, 27)
[READ] LIANA_condition_specific_HVG8000_r0_6_canonical_run_summary.tsv: (22, 2)
[

,check,match,observed,expected
0,global_celltypes_exact,True,9,9
1,condition_celltypes_exact,True,8,8
2,canonical_labels_only,True,,
3,global_cell_counts_match,True,True,True
4,global_prioritized_rows,True,1384,1384
5,global_unique_lr_pairs,True,264,264
6,global_consistency_all_match,True,True,True
7,condition_set_exact,True,GBM;LGG;ndGBM;rGBM,GBM;LGG;ndGBM;rGBM
8,condition_runs_completed,True,4,4
9,condition_prioritized_rows,True,4300,4300


[SAVED] G:\Repository\glioma-cnv-single-cell-atlas\metadata\LIANA_final_prioritization_HVG8000_r0_6_readiness.tsv shape=(10, 2)


,item,value
0,postprocessing_only,True
1,liana_rerun,False
2,global_input_notebook,27
3,condition_input_notebook,28
4,global_cell_counts_rows,9
5,global_prioritized_lr_pairs,264
6,condition_specific_runs,4
7,condition_specific_prioritized_rows,4300
8,condition_presence_rows,2450
9,canonical_input_audit_all_match,True


,condition,status,reason,n_cells,n_genes,n_celltypes,included_celltypes,n_liana_rows,n_top_exported,n_prioritized,rank_col
0,GBM,completed,NaN,93563,15176,8,Malignant_AC_like_reactive_malignant;Malignant...,11094,500,662,magnitude_rank
1,LGG,completed,NaN,6333,15176,6,Malignant_AC_like_reactive_malignant;Malignant...,11050,500,625,magnitude_rank
2,ndGBM,completed,NaN,49830,15176,8,Malignant_AC_like_reactive_malignant;Malignant...,23603,500,1337,magnitude_rank
3,rGBM,completed,NaN,33099,15176,8,Malignant_AC_like_reactive_malignant;Malignant...,30423,500,1676,magnitude_rank


## 2. Define candidate ligand–receptor axes for thesis-focused prioritization

In [4]:
# Candidate LR axes selected from global and condition-specific LIANA summaries.
# These are not claimed as functionally proven interactions; they are prioritized candidates.

CANDIDATE_LR_PATTERNS = {
    'PTN_PTPRZ1_NCL_SDC3_axis': [
        ('PTN', 'PTPRZ1'), ('PTN', 'NCL'), ('PTN', 'SDC3'), ('PTN', 'ALK'), ('PTN', 'SDC4')
    ],
    'MDK_PTPRZ1_NCL_axis': [
        ('MDK', 'PTPRZ1'), ('MDK', 'NCL'), ('MDK', 'SDC3'), ('MDK', 'SDC4'), ('MDK', 'LRP1')
    ],
    'TIMP1_CD63_axis': [('TIMP1', 'CD63')],
    'LGALS1_ITGB1_axis': [('LGALS1', 'ITGB1')],
    'MIF_EGFR_CD74_axis': [('MIF', 'EGFR'), ('MIF', 'CD74'), ('MIF', 'CXCR4'), ('MIF', 'CD44')],
    'VIM_CD44_axis': [('VIM', 'CD44')],
    'VEGFA_angiogenesis_axis': [('VEGFA', 'FLT1'), ('VEGFA', 'KDR'), ('VEGFA', 'NRP1'), ('VEGFA', 'ITGB1')],
    'APP_CD74_axis': [('APP', 'CD74')],
    'HLA_A_APLP2_cautious_axis': [('HLA-A', 'APLP2')],
}

CAUTION_AXES = {'HLA_A_APLP2_cautious_axis'}


def infer_candidate_axis(ligand, receptor, lr_pair=None):
    ligand = str(ligand) if pd.notna(ligand) else ''
    receptor = str(receptor) if pd.notna(receptor) else ''
    lr_pair = str(lr_pair) if lr_pair is not None and pd.notna(lr_pair) else f'{ligand}→{receptor}'
    hits = []
    for axis, pairs in CANDIDATE_LR_PATTERNS.items():
        for lig, rec in pairs:
            if ligand == lig and receptor == rec:
                hits.append(axis)
            elif lr_pair == f'{lig}→{rec}':
                hits.append(axis)
    if hits:
        return ';'.join(sorted(set(hits)))
    return ''


def is_candidate_row(row):
    axis = infer_candidate_axis(
        row.get('ligand', row.get('ligand_complex', '')),
        row.get('receptor', row.get('receptor_complex', '')),
        row.get('lr_pair', None)
    )
    return axis

candidate_inventory = []
for axis, pairs in CANDIDATE_LR_PATTERNS.items():
    for lig, rec in pairs:
        candidate_inventory.append({
            'candidate_axis': axis,
            'ligand': lig,
            'receptor': rec,
            'lr_pair': f'{lig}→{rec}',
            'interpretation_caution': 'high' if axis in CAUTION_AXES else 'standard'
        })
candidate_inventory = pd.DataFrame(candidate_inventory)
if len(candidate_inventory) != 23 or candidate_inventory['lr_pair'].duplicated().any():
    raise AssertionError('Candidate LR inventory drift or duplicate pair detected.')
write_tsv(candidate_inventory, METADATA_DIR / f'{OUT_PREFIX}_candidate_lr_axis_inventory.tsv')
display(candidate_inventory)

[SAVED] G:\Repository\glioma-cnv-single-cell-atlas\metadata\LIANA_final_prioritization_HVG8000_r0_6_candidate_lr_axis_inventory.tsv shape=(23, 5)


,candidate_axis,ligand,receptor,lr_pair,interpretation_caution
0,PTN_PTPRZ1_NCL_SDC3_axis,PTN,PTPRZ1,PTN→PTPRZ1,standard
1,PTN_PTPRZ1_NCL_SDC3_axis,PTN,NCL,PTN→NCL,standard
2,PTN_PTPRZ1_NCL_SDC3_axis,PTN,SDC3,PTN→SDC3,standard
3,PTN_PTPRZ1_NCL_SDC3_axis,PTN,ALK,PTN→ALK,standard
4,PTN_PTPRZ1_NCL_SDC3_axis,PTN,SDC4,PTN→SDC4,standard
5,MDK_PTPRZ1_NCL_axis,MDK,PTPRZ1,MDK→PTPRZ1,standard
6,MDK_PTPRZ1_NCL_axis,MDK,NCL,MDK→NCL,standard
7,MDK_PTPRZ1_NCL_axis,MDK,SDC3,MDK→SDC3,standard
8,MDK_PTPRZ1_NCL_axis,MDK,SDC4,MDK→SDC4,standard
9,MDK_PTPRZ1_NCL_axis,MDK,LRP1,MDK→LRP1,standard


## 3. Global LIANA summary for final reporting

In [5]:

# Rank global biological axes for thesis reporting.

gbio = global_bio_axis.copy()
if not gbio.empty:
    if 'n_interactions' not in gbio.columns and 'n_prioritized_interactions' in gbio.columns:
        gbio = gbio.rename(columns={'n_prioritized_interactions': 'n_interactions'})
    gbio['axis_display'] = gbio['biological_axis'].astype(str)
    sort_cols = [c for c in ['n_interactions', 'n_source_target_pairs'] if c in gbio.columns]
    if sort_cols:
        gbio = gbio.sort_values(sort_cols, ascending=[False] * len(sort_cols))

global_top_axes = gbio.head(30).copy()
write_tsv(global_top_axes, METADATA_DIR / f'{OUT_PREFIX}_global_top_biological_axes.tsv')

gaxis = global_axis.copy()
if not gaxis.empty:
    gaxis['axis_display'] = gaxis['communication_axis'].map(axis_display_name)
write_tsv(gaxis, METADATA_DIR / f'{OUT_PREFIX}_global_communication_axis_summary.tsv')

# Global candidate LR pairs.
glr = global_lr_summary.copy()
if not glr.empty:
    if 'lr_pair' not in glr.columns:
        glr['lr_pair'] = glr.apply(clean_lr_pair, axis=1)
    glr['candidate_axis'] = glr.apply(is_candidate_row, axis=1)
    glr_candidates = glr[glr['candidate_axis'].astype(str).str.len() > 0].copy()
else:
    glr_candidates = pd.DataFrame()

write_tsv(glr_candidates, METADATA_DIR / f'{OUT_PREFIX}_global_candidate_lr_summary.tsv')

display(global_top_axes.head(15))
display(glr_candidates.head(30))


[SAVED] G:\Repository\glioma-cnv-single-cell-atlas\metadata\LIANA_final_prioritization_HVG8000_r0_6_global_top_biological_axes.tsv shape=(25, 8)
[SAVED] G:\Repository\glioma-cnv-single-cell-atlas\metadata\LIANA_final_prioritization_HVG8000_r0_6_global_communication_axis_summary.tsv shape=(4, 6)
[SAVED] G:\Repository\glioma-cnv-single-cell-atlas\metadata\LIANA_final_prioritization_HVG8000_r0_6_global_candidate_lr_summary.tsv shape=(14, 9)


,biological_axis,source_lineage_simplified,target_lineage_simplified,n_interactions,n_source_target_pairs,median_primary_rank,min_primary_rank,axis_display
0,microenvironment_to_malignant,mural_pericyte_fibroblast,malignant_state,69,5,0.013717,2.542656e-07,microenvironment_to_malignant
1,microenvironment_to_microenvironment,mural_pericyte_fibroblast,mural_pericyte_fibroblast,63,1,0.013817,1.430284e-07,microenvironment_to_microenvironment
2,malignant_state_to_malignant_state,malignant_state,malignant_state,60,25,0.012463,1.589294e-08,malignant_state_to_malignant_state
3,microenvironment_to_microenvironment,mural_pericyte_fibroblast,endothelial,54,1,0.017783,5.146859e-06,microenvironment_to_microenvironment
4,malignant_to_microenvironment,malignant_state,mural_pericyte_fibroblast,43,5,0.017740,2.668609e-05,malignant_to_microenvironment
5,malignant_to_microenvironment,malignant_state,endothelial,38,5,0.019379,8.685173e-05,malignant_to_microenvironment
6,microenvironment_to_malignant,endothelial,malignant_state,36,5,0.025268,3.513761e-04,microenvironment_to_malignant
7,microenvironment_to_microenvironment,endothelial,endothelial,30,1,0.022652,1.157751e-05,microenvironment_to_microenvironment
8,microenvironment_to_microenvironment,mural_pericyte_fibroblast,oligodendrocyte_reference,29,1,0.021864,7.786231e-07,microenvironment_to_microenvironment
10,microenvironment_to_malignant,oligodendrocyte_reference,malignant_state,27,5,0.016009,1.835525e-05,microenvironment_to_malignant


,ligand_complex,receptor_complex,lr_pair,n_source_target_pairs,n_sources,n_targets,median_primary_rank,min_primary_rank,candidate_axis
0,TIMP1,CD63,TIMP1→CD63,68,9,9,0.000642,1.430284e-07,TIMP1_CD63_axis
1,HLA-A,APLP2,HLA-A→APLP2,64,9,8,0.007831,3.608520e-04,HLA_A_APLP2_cautious_axis
2,PTN,NCL,PTN→NCL,63,7,9,0.004415,1.245064e-05,PTN_PTPRZ1_NCL_SDC3_axis
4,PTN,PTPRZ1,PTN→PTPRZ1,49,8,8,0.000324,1.589294e-08,PTN_PTPRZ1_NCL_SDC3_axis
5,LGALS1,ITGB1,LGALS1→ITGB1,42,9,9,0.009852,4.066882e-06,LGALS1_ITGB1_axis
6,MIF,EGFR,MIF→EGFR,37,9,6,0.016187,1.091229e-03,MIF_EGFR_CD74_axis
7,VIM,CD44,VIM→CD44,30,7,5,0.003378,2.131663e-04,VIM_CD44_axis
8,MDK,PTPRZ1,MDK→PTPRZ1,27,8,5,0.003094,5.155514e-05,MDK_PTPRZ1_NCL_axis
9,PTN,SDC3,PTN→SDC3,27,7,5,0.014607,7.714426e-04,PTN_PTPRZ1_NCL_SDC3_axis
13,APP,CD74,APP→CD74,20,8,6,0.013956,1.335547e-05,APP_CD74_axis


## 4. Condition-specific comparison summaries

In [6]:

# Condition axis comparison table.

caxis = cond_axis.copy()
if not caxis.empty:
    caxis['axis_display'] = caxis['axis'].map(axis_display_name)
    axis_pivot = caxis.pivot_table(
        index='axis_display',
        columns='condition',
        values='n_interactions',
        aggfunc='sum',
        fill_value=0
    ).reset_index()
    axis_pivot['max_interactions'] = axis_pivot.drop(columns=['axis_display']).max(axis=1)
    axis_pivot = axis_pivot.sort_values('max_interactions', ascending=False)
else:
    axis_pivot = pd.DataFrame()

write_tsv(caxis, METADATA_DIR / f'{OUT_PREFIX}_condition_axis_long.tsv')
write_tsv(axis_pivot, METADATA_DIR / f'{OUT_PREFIX}_condition_axis_comparison.tsv')

# Biological axis condition comparison.
cbio = cond_bio_axis.copy()
if not cbio.empty:
    # keep top condition-specific biological axes for thesis. Top by max interactions across conditions.
    bio_pivot = cbio.pivot_table(
        index='biological_axis',
        columns='condition',
        values='n_interactions',
        aggfunc='sum',
        fill_value=0
    ).reset_index()
    condition_cols = [c for c in bio_pivot.columns if c != 'biological_axis']
    bio_pivot['max_interactions'] = bio_pivot[condition_cols].max(axis=1) if condition_cols else 0
    bio_pivot['total_interactions'] = bio_pivot[condition_cols].sum(axis=1) if condition_cols else 0
    bio_pivot = bio_pivot.sort_values(['max_interactions', 'total_interactions'], ascending=False)
else:
    bio_pivot = pd.DataFrame()

write_tsv(bio_pivot, METADATA_DIR / f'{OUT_PREFIX}_condition_biological_axis_comparison.tsv')

display(axis_pivot.head(20))
display(bio_pivot.head(30))


[SAVED] G:\Repository\glioma-cnv-single-cell-atlas\metadata\LIANA_final_prioritization_HVG8000_r0_6_condition_axis_long.tsv shape=(16, 8)
[SAVED] G:\Repository\glioma-cnv-single-cell-atlas\metadata\LIANA_final_prioritization_HVG8000_r0_6_condition_axis_comparison.tsv shape=(4, 6)
[SAVED] G:\Repository\glioma-cnv-single-cell-atlas\metadata\LIANA_final_prioritization_HVG8000_r0_6_condition_biological_axis_comparison.tsv shape=(64, 7)


condition,axis_display,GBM,LGG,ndGBM,rGBM,max_interactions
0,Malignant/cycling → malignant/cycling,292,292,427,556,556
2,Microenvironment → malignant/cycling,155,122,336,412,412
1,Malignant/cycling → microenvironment,107,138,267,361,361
3,Microenvironment → microenvironment,108,73,307,347,347


condition,biological_axis,GBM,LGG,ndGBM,rGBM,max_interactions,total_interactions
48,mural_pericyte_fibroblast_to_endothelial,21,0,67,73,73,161
0,endothelial_to_endothelial,7,16,38,65,65,126
54,mural_pericyte_fibroblast_to_mural_pericyte_fi...,24,0,64,43,64,131
49,mural_pericyte_fibroblast_to_malignant_AC_reac...,22,0,46,53,53,121
8,malignant_AC_reactive_to_endothelial,12,23,21,44,44,100
9,malignant_AC_reactive_to_malignant_AC_reactive,13,20,23,39,39,95
32,malignant_cycling_to_endothelial,10,15,25,38,38,88
5,endothelial_to_malignant_hypoxia_stress,9,0,12,37,37,58
6,endothelial_to_mural_pericyte_fibroblast,11,0,36,26,36,73
33,malignant_cycling_to_malignant_AC_reactive,13,19,32,35,35,99


## 5. Candidate LR condition presence and source-target details

In [7]:

# Candidate interaction presence across conditions.

pres = cond_presence.copy()
if not pres.empty:
    if 'lr_pair' not in pres.columns:
        pres['lr_pair'] = pres.apply(clean_lr_pair, axis=1)
    pres['candidate_axis'] = pres.apply(is_candidate_row, axis=1)
    pres_candidates = pres[pres['candidate_axis'].astype(str).str.len() > 0].copy()
    # Add caution flag
    pres_candidates['interpretation_caution'] = pres_candidates['candidate_axis'].apply(
        lambda x: 'high' if any(ax in CAUTION_AXES for ax in str(x).split(';')) else 'standard'
    )
    sort_cols = [c for c in ['n_conditions', 'min_rank_all', 'median_rank_all'] if c in pres_candidates.columns]
    if sort_cols:
        asc = [False if c == 'n_conditions' else True for c in sort_cols]
        pres_candidates = pres_candidates.sort_values(sort_cols, ascending=asc)
else:
    pres_candidates = pd.DataFrame()

write_tsv(pres_candidates, METADATA_DIR / f'{OUT_PREFIX}_candidate_lr_condition_presence.tsv')

# Candidate rows from condition-specific prioritized interactions.
cp = cond_prioritized.copy()
if not cp.empty:
    if 'lr_pair' not in cp.columns:
        cp['lr_pair'] = cp.apply(clean_lr_pair, axis=1)
    cp['candidate_axis'] = cp.apply(is_candidate_row, axis=1)
    cp_candidates = cp[cp['candidate_axis'].astype(str).str.len() > 0].copy()
    # Keep a compact table for upload/reading.
    keep_cols = [c for c in [
        'condition', 'candidate_axis', 'ligand', 'receptor', 'lr_pair',
        'source_clean', 'target_clean', 'source_target', 'axis', 'biological_axis',
        'rank_for_sorting', 'magnitude_rank', 'specificity_rank', 'lr_means', 'expr_prod',
        'scaled_weight', 'lr_logfc', 'prioritization_rule'
    ] if c in cp_candidates.columns]
    cp_candidates_compact = cp_candidates[keep_cols].copy()
    sort_cols = [c for c in ['candidate_axis', 'condition', 'rank_for_sorting'] if c in cp_candidates_compact.columns]
    if sort_cols:
        cp_candidates_compact = cp_candidates_compact.sort_values(sort_cols)
else:
    cp_candidates_compact = pd.DataFrame()

write_tsv(cp_candidates_compact, METADATA_DIR / f'{OUT_PREFIX}_candidate_lr_source_target_details.tsv')

display(pres_candidates.head(40))
display(cp_candidates_compact.head(40))


[SAVED] G:\Repository\glioma-cnv-single-cell-atlas\metadata\LIANA_final_prioritization_HVG8000_r0_6_candidate_lr_condition_presence.tsv shape=(548, 18)
[SAVED] G:\Repository\glioma-cnv-single-cell-atlas\metadata\LIANA_final_prioritization_HVG8000_r0_6_candidate_lr_source_target_details.tsv shape=(1298, 18)


,interaction_key,source_clean,target_clean,lr_pair,ligand,receptor,axis,biological_axis,n_conditions,conditions,min_rank_all,median_rank_all,rank_GBM,rank_LGG,rank_ndGBM,rank_rGBM,candidate_axis,interpretation_caution
0,Malignant_AC_like_reactive_malignant|Malignant...,Malignant_AC_like_reactive_malignant,Malignant_OPC_NPC_like_malignant,PTN→PTPRZ1,PTN,PTPRZ1,malignant_or_cycling_to_malignant_or_cycling,malignant_AC_reactive_to_malignant_OPC_NPC,4,GBM;LGG;ndGBM;rGBM,9.723642e-09,7.450967e-06,0.000014,3.246430e-05,5.814829e-07,9.723642e-09,PTN_PTPRZ1_NCL_SDC3_axis,standard
1,Malignant_cycling_candidate|Malignant_OPC_NPC_...,Malignant_cycling_candidate,Malignant_OPC_NPC_like_malignant,PTN→PTPRZ1,PTN,PTPRZ1,malignant_or_cycling_to_malignant_or_cycling,malignant_cycling_to_malignant_OPC_NPC,4,GBM;LGG;ndGBM;rGBM,1.615459e-08,1.668462e-07,0.000004,2.947987e-07,1.615459e-08,3.889372e-08,PTN_PTPRZ1_NCL_SDC3_axis,standard
2,Malignant_cycling_candidate|Malignant_AC_like_...,Malignant_cycling_candidate,Malignant_AC_like_reactive_malignant,PTN→PTPRZ1,PTN,PTPRZ1,malignant_or_cycling_to_malignant_or_cycling,malignant_cycling_to_malignant_AC_reactive,4,GBM;LGG;ndGBM;rGBM,6.461652e-08,2.673237e-05,0.000053,4.931645e-04,6.461652e-08,2.430697e-07,PTN_PTPRZ1_NCL_SDC3_axis,standard
3,Malignant_OPC_NPC_like_malignant|Malignant_OPC...,Malignant_OPC_NPC_like_malignant,Malignant_OPC_NPC_like_malignant,PTN→PTPRZ1,PTN,PTPRZ1,malignant_or_cycling_to_malignant_or_cycling,malignant_OPC_NPC_to_malignant_OPC_NPC,4,GBM;LGG;ndGBM;rGBM,8.750894e-08,6.575513e-07,0.000001,1.244698e-05,1.453830e-07,8.750894e-08,PTN_PTPRZ1_NCL_SDC3_axis,standard
4,Malignant_AC_like_reactive_malignant|Malignant...,Malignant_AC_like_reactive_malignant,Malignant_AC_like_reactive_malignant,PTN→PTPRZ1,PTN,PTPRZ1,malignant_or_cycling_to_malignant_or_cycling,malignant_AC_reactive_to_malignant_AC_reactive,4,GBM;LGG;ndGBM;rGBM,1.555680e-07,9.164851e-05,0.000182,1.124864e-03,1.033689e-06,1.555680e-07,PTN_PTPRZ1_NCL_SDC3_axis,standard
5,Malignant_OPC_NPC_like_malignant|Malignant_AC_...,Malignant_OPC_NPC_like_malignant,Malignant_AC_like_reactive_malignant,PTN→PTPRZ1,PTN,PTPRZ1,malignant_or_cycling_to_malignant_or_cycling,malignant_OPC_NPC_to_malignant_AC_reactive,4,GBM;LGG;ndGBM;rGBM,2.584515e-07,1.627873e-05,0.000032,9.513280e-04,2.584515e-07,3.500127e-07,PTN_PTPRZ1_NCL_SDC3_axis,standard
6,Malignant_AC_like_reactive_malignant|Malignant...,Malignant_AC_like_reactive_malignant,Malignant_cycling_candidate,PTN→PTPRZ1,PTN,PTPRZ1,malignant_or_cycling_to_malignant_or_cycling,malignant_AC_reactive_to_malignant_cycling,4,GBM;LGG;ndGBM;rGBM,4.763958e-07,2.418185e-05,0.000046,4.463326e-04,2.729199e-06,4.763958e-07,PTN_PTPRZ1_NCL_SDC3_axis,standard
7,Malignant_cycling_candidate|Malignant_cycling_...,Malignant_cycling_candidate,Malignant_cycling_candidate,PTN→PTPRZ1,PTN,PTPRZ1,malignant_or_cycling_to_malignant_or_cycling,malignant_cycling_to_malignant_cycling,4,GBM;LGG;ndGBM;rGBM,6.222176e-07,3.791100e-06,0.000019,5.967152e-06,1.615048e-06,6.222176e-07,PTN_PTPRZ1_NCL_SDC3_axis,standard
8,Malignant_OPC_NPC_like_malignant|Malignant_cyc...,Malignant_OPC_NPC_like_malignant,Malignant_cycling_candidate,PTN→PTPRZ1,PTN,PTPRZ1,malignant_or_cycling_to_malignant_or_cycling,malignant_OPC_NPC_to_malignant_cycling,4,GBM;LGG;ndGBM;rGBM,7.874769e-07,4.631132e-06,0.000007,2.470880e-04,1.954153e-06,7.874769e-07,PTN_PTPRZ1_NCL_SDC3_axis,standard
9,Oligodendrocyte_myelinating|Malignant_OPC_NPC_...,Oligodendrocyte_myelinating,Malignant_OPC_NPC_like_malignant,PTN→PTPRZ1,PTN,PTPRZ1,microenvironment_to_malignant_or_cycling,oligodendrocyte_reference_to_malignant_OPC_NPC,4,GBM;LGG;ndGBM;rGBM,1.176303e-06,3.370071e-05,0.001401,3.893772e-05,2.846369e-05,1.176303e-06,PTN_PTPRZ1_NCL_SDC3_axis,standard


,condition,candidate_axis,ligand,receptor,lr_pair,source_clean,target_clean,source_target,axis,biological_axis,rank_for_sorting,magnitude_rank,specificity_rank,lr_means,expr_prod,scaled_weight,lr_logfc,prioritization_rule
218,GBM,APP_CD74_axis,APP,CD74,APP→CD74,Oligodendrocyte_myelinating,Oligodendrocyte_myelinating,Oligodendrocyte_myelinating → Oligodendrocyte_...,microenvironment_to_microenvironment,oligodendrocyte_reference_to_oligodendrocyte_r...,0.009528,0.009528,0.051279,1.679378,2.798220,0.619629,0.924317,magnitude_rank<=0.05
250,GBM,APP_CD74_axis,APP,CD74,APP→CD74,Endothelial,Oligodendrocyte_myelinating,Endothelial → Oligodendrocyte_myelinating,microenvironment_to_microenvironment,endothelial_to_oligodendrocyte_reference,0.011250,0.011250,0.051279,1.640296,2.678572,0.583739,1.069985,magnitude_rank<=0.05
300,GBM,APP_CD74_axis,APP,CD74,APP→CD74,Malignant_AC_like_reactive_malignant,Oligodendrocyte_myelinating,Malignant_AC_like_reactive_malignant → Oligode...,malignant_or_cycling_to_microenvironment,malignant_AC_reactive_to_oligodendrocyte_refer...,0.014471,0.014471,0.051279,1.569073,2.460520,0.518338,0.879106,magnitude_rank<=0.05
446,GBM,APP_CD74_axis,APP,CD74,APP→CD74,Malignant_OPC_NPC_like_malignant,Oligodendrocyte_myelinating,Malignant_OPC_NPC_like_malignant → Oligodendro...,malignant_or_cycling_to_microenvironment,malignant_OPC_NPC_to_oligodendrocyte_reference,0.025960,0.025960,0.051279,1.422412,2.011518,0.383658,0.673789,magnitude_rank<=0.05
515,GBM,APP_CD74_axis,APP,CD74,APP→CD74,Malignant_hypoxia_stress_malignant,Oligodendrocyte_myelinating,Malignant_hypoxia_stress_malignant → Oligodend...,malignant_or_cycling_to_microenvironment,malignant_hypoxia_stress_to_oligodendrocyte_re...,0.032338,0.032338,0.055935,1.385763,1.899317,0.350004,0.491444,magnitude_rank<=0.05
519,GBM,APP_CD74_axis,APP,CD74,APP→CD74,Malignant_NPC_neuronal_like_malignant,Oligodendrocyte_myelinating,Malignant_NPC_neuronal_like_malignant → Oligod...,malignant_or_cycling_to_microenvironment,malignant_NPC_neuronal_to_oligodendrocyte_refe...,0.032994,0.032994,0.057177,1.382358,1.888893,0.346880,0.610633,magnitude_rank<=0.05
1151,LGG,APP_CD74_axis,APP,CD74,APP→CD74,Oligodendrocyte_myelinating,Endothelial,Oligodendrocyte_myelinating → Endothelial,microenvironment_to_microenvironment,oligodendrocyte_reference_to_endothelial,0.032309,0.032309,0.012907,1.327068,1.359364,0.819571,0.929826,magnitude_rank<=0.05
1918,ndGBM,APP_CD74_axis,APP,CD74,APP→CD74,Oligodendrocyte_myelinating,Endothelial,Oligodendrocyte_myelinating → Endothelial,microenvironment_to_microenvironment,oligodendrocyte_reference_to_endothelial,0.013490,0.013490,0.047630,1.477907,1.995908,1.097149,1.411193,magnitude_rank<=0.05
1968,ndGBM,APP_CD74_axis,APP,CD74,APP→CD74,Endothelial,Endothelial,Endothelial → Endothelial,microenvironment_to_microenvironment,endothelial_to_endothelial,0.015544,0.015544,0.047630,1.449127,1.935818,1.061098,1.457364,magnitude_rank<=0.05
2230,ndGBM,APP_CD74_axis,APP,CD74,APP→CD74,Oligodendrocyte_myelinating,Oligodendrocyte_myelinating,Oligodendrocyte_myelinating → Oligodendrocyte_...,microenvironment_to_microenvironment,oligodendrocyte_reference_to_oligodendrocyte_r...,0.026778,0.026778,0.047630,1.322576,1.401974,0.860877,1.283906,magnitude_rank<=0.05


## 6. Build thesis-ready interpretation table

In [8]:
# Create a compact interpretive table combining biological axis, recurrence, and caution.

interpretation_rows = []

# Use candidate presence to build concise summaries.
if not pres_candidates.empty:
    for axis, sub in pres_candidates.groupby('candidate_axis'):
        lr_pairs = '; '.join(sorted(sub['lr_pair'].dropna().astype(str).unique())[:12])
        n_keys = len(sub)
        conditions = sorted(set(';'.join(sub.get('conditions', pd.Series(dtype=str)).dropna().astype(str)).replace(',', ';').split(';')))
        conditions = [c for c in conditions if c]
        n_cond_max = sub['n_conditions'].max() if 'n_conditions' in sub.columns else np.nan
        top_bio = '; '.join(sub.get('biological_axis', pd.Series(dtype=str)).dropna().astype(str).value_counts().head(5).index.tolist())
        interpretation_rows.append({
            'candidate_axis': axis,
            'n_candidate_interaction_keys': n_keys,
            'max_n_conditions_detected': n_cond_max,
            'conditions_observed': ';'.join(conditions),
            'representative_lr_pairs': lr_pairs,
            'dominant_biological_axes': top_bio,
            'interpretation_caution': 'high' if axis in CAUTION_AXES else 'standard',
            'suggested_thesis_interpretation': {
                'PTN_PTPRZ1_NCL_SDC3_axis': 'Candidate malignant-state communication axis linked to glioma-associated growth/progenitor signaling; prioritize as recurrent malignant/microenvironment communication candidate.',
                'MDK_PTPRZ1_NCL_axis': 'Candidate growth-factor-like communication axis, often related to malignant and stromal communication; interpret as putative signaling candidate.',
                'TIMP1_CD63_axis': 'Candidate ECM/remodeling and stress-associated communication axis, prominent across multiple sender-target contexts.',
                'LGALS1_ITGB1_axis': 'Candidate adhesion/ECM-associated communication axis connecting malignant and microenvironmental compartments.',
                'MIF_EGFR_CD74_axis': 'Candidate inflammatory/growth signaling axis; interpret with caution because MIF can act through multiple receptor complexes.',
                'VIM_CD44_axis': 'Candidate mesenchymal/stress-associated communication axis, consistent with reactive/MES-like programs.',
                'VEGFA_angiogenesis_axis': 'Candidate angiogenic communication axis, especially relevant when involving endothelial or mural/pericyte-like targets.',
                'APP_CD74_axis': 'Candidate immune/inflammatory communication axis; interpret as prioritized LR signal rather than direct functional proof.',
                'HLA_A_APLP2_cautious_axis': 'Cautious candidate; HLA-related interactions may reflect broad expression/antigen-presentation signals and should not be overinterpreted.'
            }.get(axis, 'Candidate LIANA-prioritized ligand–receptor axis.')
        })

interpretation = pd.DataFrame(interpretation_rows)
if not interpretation.empty:
    interpretation = interpretation.sort_values(['max_n_conditions_detected', 'n_candidate_interaction_keys'], ascending=False)

write_tsv(interpretation, METADATA_DIR / f'{OUT_PREFIX}_thesis_candidate_axis_interpretation.tsv')

historical_checkpoints = pd.DataFrame([
    {'checkpoint': 'candidate_inventory_rows', 'expected_historical': 23, 'observed': len(candidate_inventory), 'match': len(candidate_inventory) == 23},
    {'checkpoint': 'global_top_biological_axes_rows', 'expected_historical': 25, 'observed': len(global_top_axes), 'match': len(global_top_axes) == 25},
    {'checkpoint': 'global_candidate_lr_rows', 'expected_historical': 14, 'observed': len(glr_candidates), 'match': len(glr_candidates) == 14},
    {'checkpoint': 'condition_axis_long_rows', 'expected_historical': 16, 'observed': len(caxis), 'match': len(caxis) == 16},
    {'checkpoint': 'condition_axis_comparison_rows', 'expected_historical': 4, 'observed': len(axis_pivot), 'match': len(axis_pivot) == 4},
    {'checkpoint': 'condition_biological_axis_rows', 'expected_historical': 64, 'observed': len(bio_pivot), 'match': len(bio_pivot) == 64},
    {'checkpoint': 'candidate_presence_rows', 'expected_historical': 548, 'observed': len(pres_candidates), 'match': len(pres_candidates) == 548},
    {'checkpoint': 'candidate_source_target_rows', 'expected_historical': 1298, 'observed': len(cp_candidates_compact), 'match': len(cp_candidates_compact) == 1298},
    {'checkpoint': 'candidate_axes_prioritized', 'expected_historical': 9, 'observed': interpretation['candidate_axis'].nunique(), 'match': interpretation['candidate_axis'].nunique() == 9},
])
write_tsv(historical_checkpoints, METADATA_DIR / f'{OUT_PREFIX}_historical_checkpoint_audit.tsv')
display(interpretation)
display(historical_checkpoints)
if not historical_checkpoints['match'].all():
    raise AssertionError('Notebook 29 outputs drift from historical checkpoints.')

[SAVED] G:\Repository\glioma-cnv-single-cell-atlas\metadata\LIANA_final_prioritization_HVG8000_r0_6_thesis_candidate_axis_interpretation.tsv shape=(9, 8)
[SAVED] G:\Repository\glioma-cnv-single-cell-atlas\metadata\LIANA_final_prioritization_HVG8000_r0_6_historical_checkpoint_audit.tsv shape=(9, 4)


,candidate_axis,n_candidate_interaction_keys,max_n_conditions_detected,conditions_observed,representative_lr_pairs,dominant_biological_axes,interpretation_caution,suggested_thesis_interpretation
5,PTN_PTPRZ1_NCL_SDC3_axis,162,4,GBM;LGG;ndGBM;rGBM,PTN→NCL; PTN→PTPRZ1; PTN→SDC3; PTN→SDC4,malignant_cycling_to_malignant_hypoxia_stress;...,standard,Candidate malignant-state communication axis l...
3,MDK_PTPRZ1_NCL_axis,110,4,GBM;LGG;ndGBM;rGBM,MDK→LRP1; MDK→NCL; MDK→PTPRZ1; MDK→SDC3,malignant_cycling_to_malignant_OPC_NPC; malign...,standard,Candidate growth-factor-like communication axi...
6,TIMP1_CD63_axis,64,4,GBM;LGG;ndGBM;rGBM,TIMP1→CD63,endothelial_to_malignant_AC_reactive; endothel...,standard,Candidate ECM/remodeling and stress-associated...
1,HLA_A_APLP2_cautious_axis,57,4,GBM;LGG;ndGBM;rGBM,HLA-A→APLP2,endothelial_to_endothelial; endothelial_to_mal...,high,Cautious candidate; HLA-related interactions m...
2,LGALS1_ITGB1_axis,46,4,GBM;LGG;ndGBM;rGBM,LGALS1→ITGB1,endothelial_to_endothelial; mural_pericyte_fib...,standard,Candidate adhesion/ECM-associated communicatio...
4,MIF_EGFR_CD74_axis,45,4,GBM;LGG;ndGBM;rGBM,MIF→EGFR,oligodendrocyte_reference_to_malignant_AC_reac...,standard,Candidate inflammatory/growth signaling axis; ...
8,VIM_CD44_axis,31,4,GBM;LGG;ndGBM;rGBM,VIM→CD44,malignant_AC_reactive_to_malignant_AC_reactive...,standard,Candidate mesenchymal/stress-associated commun...
0,APP_CD74_axis,17,3,GBM;LGG;ndGBM;rGBM,APP→CD74,oligodendrocyte_reference_to_endothelial; olig...,standard,Candidate immune/inflammatory communication ax...
7,VEGFA_angiogenesis_axis,16,3,GBM;ndGBM;rGBM,VEGFA→ITGB1; VEGFA→NRP1,malignant_hypoxia_stress_to_mural_pericyte_fib...,standard,"Candidate angiogenic communication axis, espec..."


,checkpoint,expected_historical,observed,match
0,candidate_inventory_rows,23,23,True
1,global_top_biological_axes_rows,25,25,True
2,global_candidate_lr_rows,14,14,True
3,condition_axis_long_rows,16,16,True
4,condition_axis_comparison_rows,4,4,True
5,condition_biological_axis_rows,64,64,True
6,candidate_presence_rows,548,548,True
7,candidate_source_target_rows,1298,1298,True
8,candidate_axes_prioritized,9,9,True


## 7. Figures: global axes, condition axes, and candidate LR presence

In [9]:

figure_records = []

# Figure 1: Global biological axes barplot
if not global_top_axes.empty and 'n_interactions' in global_top_axes.columns:
    dfp = global_top_axes.head(15).copy()
    labels = dfp['biological_axis'].astype(str).str.replace('_', ' ', regex=False)
    plt.figure(figsize=(9, max(4, 0.35 * len(dfp))))
    plt.barh(range(len(dfp)), dfp['n_interactions'])
    plt.yticks(range(len(dfp)), labels)
    plt.gca().invert_yaxis()
    plt.xlabel('Number of prioritized LIANA interactions')
    plt.title('Global LIANA prioritized biological axes')
    p = save_fig(FIGURE_DIR / f'{OUT_PREFIX}_global_biological_axes_barplot.png')
    figure_records.append({'figure': 'global_biological_axes_barplot', 'path': str(p)})

# Figure 2: Condition communication axis heatmap
if not axis_pivot.empty:
    dfh = axis_pivot.copy()
    condition_cols = [c for c in ['GBM', 'LGG', 'ndGBM', 'rGBM'] if c in dfh.columns]
    if condition_cols:
        plot_df = dfh.set_index('axis_display')[condition_cols]
        plt.figure(figsize=(8, max(3, 0.55 * len(plot_df))))
        plt.imshow(plot_df.values, aspect='auto')
        plt.xticks(range(len(condition_cols)), condition_cols, rotation=45, ha='right')
        plt.yticks(range(len(plot_df.index)), plot_df.index)
        plt.colorbar(label='Number of prioritized interactions')
        plt.title('Condition-specific LIANA communication axes')
        p = save_fig(FIGURE_DIR / f'{OUT_PREFIX}_condition_axis_heatmap.png')
        figure_records.append({'figure': 'condition_axis_heatmap', 'path': str(p)})

# Figure 3: Candidate LR condition presence heatmap by min rank converted to -log10(rank)
if not pres_candidates.empty:
    pc = pres_candidates.copy()
    # Use one row per interaction key. Prefer rank columns per condition if available.
    rank_cols = [c for c in ['rank_GBM', 'rank_LGG', 'rank_ndGBM', 'rank_rGBM'] if c in pc.columns]
    if rank_cols:
        pc['display_key'] = pc['lr_pair'].astype(str) + ' | ' + pc.get('source_clean', '').astype(str).str.replace('Malignant_', 'M_', regex=False).str.replace('NonMalignant_', 'NM_', regex=False) + '→' + pc.get('target_clean', '').astype(str).str.replace('Malignant_', 'M_', regex=False).str.replace('NonMalignant_', 'NM_', regex=False)
        pc = pc.sort_values(['n_conditions', 'min_rank_all'], ascending=[False, True]).head(35)
        mat = pc[rank_cols].astype(float).replace([np.inf, -np.inf], np.nan)
        score = -np.log10(mat.clip(lower=1e-12))
        score = score.fillna(0)
        labels = [c.replace('rank_', '') for c in rank_cols]
        plt.figure(figsize=(9, max(5, 0.25 * len(pc))))
        plt.imshow(score.values, aspect='auto')
        plt.xticks(range(len(labels)), labels, rotation=45, ha='right')
        plt.yticks(range(len(pc)), pc['display_key'])
        plt.colorbar(label='-log10(rank); 0 = not prioritized/detected')
        plt.title('Candidate LIANA interactions across conditions')
        p = save_fig(FIGURE_DIR / f'{OUT_PREFIX}_candidate_lr_condition_presence_heatmap.png')
        figure_records.append({'figure': 'candidate_lr_condition_presence_heatmap', 'path': str(p)})

# Figure 4: Candidate axis counts
if not interpretation.empty:
    dfp = interpretation.copy()
    plt.figure(figsize=(9, max(4, 0.45 * len(dfp))))
    plt.barh(range(len(dfp)), dfp['n_candidate_interaction_keys'])
    plt.yticks(range(len(dfp)), dfp['candidate_axis'])
    plt.gca().invert_yaxis()
    plt.xlabel('Number of condition-specific candidate interaction keys')
    plt.title('Final prioritized LIANA candidate axes')
    p = save_fig(FIGURE_DIR / f'{OUT_PREFIX}_candidate_axis_counts_barplot.png')
    figure_records.append({'figure': 'candidate_axis_counts_barplot', 'path': str(p)})

figure_manifest = pd.DataFrame(figure_records)
write_tsv(figure_manifest, METADATA_DIR / f'{OUT_PREFIX}_figure_manifest.tsv')
display(figure_manifest)


[SAVED FIG] G:\Repository\glioma-cnv-single-cell-atlas\figures\LIANA_final_prioritization_HVG8000_r0_6\LIANA_final_prioritization_HVG8000_r0_6_global_biological_axes_barplot.png
[SAVED FIG] G:\Repository\glioma-cnv-single-cell-atlas\figures\LIANA_final_prioritization_HVG8000_r0_6\LIANA_final_prioritization_HVG8000_r0_6_condition_axis_heatmap.png
[SAVED FIG] G:\Repository\glioma-cnv-single-cell-atlas\figures\LIANA_final_prioritization_HVG8000_r0_6\LIANA_final_prioritization_HVG8000_r0_6_candidate_lr_condition_presence_heatmap.png
[SAVED FIG] G:\Repository\glioma-cnv-single-cell-atlas\figures\LIANA_final_prioritization_HVG8000_r0_6\LIANA_final_prioritization_HVG8000_r0_6_candidate_axis_counts_barplot.png
[SAVED] G:\Repository\glioma-cnv-single-cell-atlas\metadata\LIANA_final_prioritization_HVG8000_r0_6_figure_manifest.tsv shape=(4, 2)


,figure,path
0,global_biological_axes_barplot,G:\Repository\glioma-cnv-single-cell-atlas\fig...
1,condition_axis_heatmap,G:\Repository\glioma-cnv-single-cell-atlas\fig...
2,candidate_lr_condition_presence_heatmap,G:\Repository\glioma-cnv-single-cell-atlas\fig...
3,candidate_axis_counts_barplot,G:\Repository\glioma-cnv-single-cell-atlas\fig...


## 8. Final summary and output manifest

In [10]:
# Final compact run summary

summary_rows = []
summary_rows.append({'item': 'postprocessing_only', 'value': True})
summary_rows.append({'item': 'liana_rerun', 'value': False})
summary_rows.append({'item': 'canonical_input_audit_all_match', 'value': bool(input_audit['match'].all())})
summary_rows.append({'item': 'historical_checkpoints_all_match', 'value': bool(historical_checkpoints['match'].all())})
summary_rows.append({'item': 'global_celltypes_used', 'value': global_cell_counts.shape[0]})
summary_rows.append({'item': 'global_top_biological_axes_saved', 'value': len(global_top_axes)})
summary_rows.append({'item': 'condition_runs_completed', 'value': int((cond_run.get('status', pd.Series(dtype=str)) == 'completed').sum()) if not cond_run.empty else 0})
summary_rows.append({'item': 'condition_prioritized_rows', 'value': len(cond_prioritized)})
summary_rows.append({'item': 'candidate_lr_condition_presence_rows', 'value': len(pres_candidates)})
summary_rows.append({'item': 'candidate_lr_source_target_detail_rows', 'value': len(cp_candidates_compact)})
summary_rows.append({'item': 'candidate_axes_prioritized', 'value': interpretation['candidate_axis'].nunique() if not interpretation.empty else 0})
summary_rows.append({'item': 'figures_created', 'value': len(figure_manifest)})
summary_rows.append({'item': 'interpretation_note', 'value': 'Candidate communication axes only; not functional proof and not replicate-aware differential communication.'})
summary_rows.append({'item': 'completed_at', 'value': datetime.now().isoformat(timespec='seconds')})
summary = pd.DataFrame(summary_rows)
write_tsv(summary, METADATA_DIR / f'{OUT_PREFIX}_run_summary.tsv')

# Final output manifest
expected_outputs = [
    METADATA_DIR / f'{OUT_PREFIX}_readiness.tsv',
    METADATA_DIR / f'{OUT_PREFIX}_input_audit.tsv',
    METADATA_DIR / f'{OUT_PREFIX}_historical_checkpoint_audit.tsv',
    METADATA_DIR / f'{OUT_PREFIX}_candidate_lr_axis_inventory.tsv',
    METADATA_DIR / f'{OUT_PREFIX}_global_top_biological_axes.tsv',
    METADATA_DIR / f'{OUT_PREFIX}_global_communication_axis_summary.tsv',
    METADATA_DIR / f'{OUT_PREFIX}_global_candidate_lr_summary.tsv',
    METADATA_DIR / f'{OUT_PREFIX}_condition_axis_long.tsv',
    METADATA_DIR / f'{OUT_PREFIX}_condition_axis_comparison.tsv',
    METADATA_DIR / f'{OUT_PREFIX}_condition_biological_axis_comparison.tsv',
    METADATA_DIR / f'{OUT_PREFIX}_candidate_lr_condition_presence.tsv',
    METADATA_DIR / f'{OUT_PREFIX}_candidate_lr_source_target_details.tsv',
    METADATA_DIR / f'{OUT_PREFIX}_thesis_candidate_axis_interpretation.tsv',
    METADATA_DIR / f'{OUT_PREFIX}_figure_manifest.tsv',
    METADATA_DIR / f'{OUT_PREFIX}_run_summary.tsv',
]
manifest = []
for p in expected_outputs:
    p = Path(p)
    manifest.append({'path': str(p), 'exists': p.exists(), 'size_mb': round(p.stat().st_size / 1024**2, 3) if p.exists() else np.nan})
for p in figure_manifest['path'].tolist() if not figure_manifest.empty else []:
    p = Path(p)
    manifest.append({'path': str(p), 'exists': p.exists(), 'size_mb': round(p.stat().st_size / 1024**2, 3) if p.exists() else np.nan})
manifest = pd.DataFrame(manifest)
write_tsv(manifest, METADATA_DIR / f'{OUT_PREFIX}_output_manifest.tsv')

display(summary)
display(manifest)
if not manifest['exists'].all():
    raise AssertionError('One or more expected Notebook 29 outputs are missing.')
if not (input_audit['match'].all() and historical_checkpoints['match'].all()):
    raise AssertionError('Notebook 29 final validation gate failed.')
print('[DONE] Notebook 29 completed successfully.')

[SAVED] G:\Repository\glioma-cnv-single-cell-atlas\metadata\LIANA_final_prioritization_HVG8000_r0_6_run_summary.tsv shape=(14, 2)
[SAVED] G:\Repository\glioma-cnv-single-cell-atlas\metadata\LIANA_final_prioritization_HVG8000_r0_6_output_manifest.tsv shape=(19, 3)


,item,value
0,postprocessing_only,True
1,liana_rerun,False
2,canonical_input_audit_all_match,True
3,historical_checkpoints_all_match,True
4,global_celltypes_used,9
5,global_top_biological_axes_saved,25
6,condition_runs_completed,4
7,condition_prioritized_rows,4300
8,candidate_lr_condition_presence_rows,548
9,candidate_lr_source_target_detail_rows,1298


,path,exists,size_mb
0,G:\Repository\glioma-cnv-single-cell-atlas\met...,True,0.000
1,G:\Repository\glioma-cnv-single-cell-atlas\met...,True,0.001
2,G:\Repository\glioma-cnv-single-cell-atlas\met...,True,0.000
3,G:\Repository\glioma-cnv-single-cell-atlas\met...,True,0.001
4,G:\Repository\glioma-cnv-single-cell-atlas\met...,True,0.004
5,G:\Repository\glioma-cnv-single-cell-atlas\met...,True,0.001
6,G:\Repository\glioma-cnv-single-cell-atlas\met...,True,0.001
7,G:\Repository\glioma-cnv-single-cell-atlas\met...,True,0.002
8,G:\Repository\glioma-cnv-single-cell-atlas\met...,True,0.000
9,G:\Repository\glioma-cnv-single-cell-atlas\met...,True,0.004


[DONE] Notebook 29 completed successfully.


## Outputs to send back

After a complete fresh-kernel run, send the Notebook 29 metadata files with prefix:

```text
metadata/LIANA_final_prioritization_HVG8000_r0_6_*
```

The two audit tables and final manifest are sufficient for the first validation pass;
figures can remain parked until their numerical source tables pass.